In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda


In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
with open("scifact/qrels/test.tsv") as f:
    all_lines=f.read().splitlines()

for i in range(5):
    print(all_lines[i])

print(len(all_lines))

In [ ]:
qrels={}
for line in all_lines:
    line=line.split("\t")
    q_id,c_id,score = line
    if q_id not in qrels:
        qrels[q_id]=[]
    qrels[q_id].append(c_id)
    


In [ ]:
qrels

In [ ]:
corpus=[]

In [ ]:
import json
with open("scifact/corpus.jsonl","r",encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))

In [ ]:
print(len(corpus))

In [ ]:
print(corpus[0])

In [ ]:
print(corpus[0]["_id"])

In [ ]:
print(corpus[0]["metadata"])

In [ ]:
for entry in corpus:
    if "corpus_id" not in entry["metadata"]:
        entry["metadata"]["corpus_id"]={}
    entry["metadata"]["corpus_id"]=entry["_id"]

In [ ]:
print(corpus[0])

In [ ]:
docs=[]
for doc in corpus:
    docs.append(
        Document(
            page_content=f'{doc["title"]}. {doc["text"]}',
            metadata={"corpus_id": doc["_id"]}
        )
    )

In [ ]:
words=len(docs[0].page_content.split())
print(words)

In [ ]:
embeddings=OllamaEmbeddings(model="mxbai-embed-large")


In [ ]:
print(len(docs))

In [ ]:
print(len(docs[0].page_content))
print(len(docs[100].page_content))

In [ ]:
DDB = "./Chroma-mxbai"
BATCH = 32

Dense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)
batch=1
if Dense_vs._collection.count() == len(docs):
    print("Loaded existing DB")
else:
    Dense_vs.delete_collection()
    for i in range(0, len(docs), BATCH):
        Dense_vs = Chroma.from_documents(
            docs[i:i + BATCH], embeddings, persist_directory=DDB
        )
        batch=batch+1
        print(f"batch {batch} embedded successfuly")
    print("Built DB:", Dense_vs._collection.count())


In [ ]:
"""
Dense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)

# corpus_ids already stored, read from the metadata
stored = Dense_vs.get(include=["metadatas"])["metadatas"]
done = {m["corpus_id"] for m in stored}

todo = [d for d in docs if d.metadata["corpus_id"] not in done]
print(f"{len(done)} stored, {len(todo)} remaining")

for i in range(0, len(todo), BATCH):
    Dense_vs.add_documents(todo[i:i + BATCH])
    print(f"{min(i + BATCH, len(todo))}/{len(todo)} embedded")

print("Total in DB:", Dense_vs._collection.count())   # expect 5183
"""


In [ ]:
"""
Dense_vs = Chroma(persist_directory=DDB, embedding_function=embeddings)

# corpus_ids already stored, read from the metadata
stored = Dense_vs.get(include=["metadatas"])["metadatas"]
done = {m["corpus_id"] for m in stored}

todo = [d for d in docs if d.metadata["corpus_id"] not in done]
print(f"{len(done)} stored, {len(todo)} remaining")

for i in range(0, len(todo), BATCH):
    Dense_vs.add_documents(todo[i:i + BATCH])
    print(f"{min(i + BATCH, len(todo))}/{len(todo)} embedded")

print("Total in DB:", Dense_vs._collection.count())   # expect 5183
"""

In [ ]:
retriever=Dense_vs.as_retriever()

In [ ]:
query=[]
with open("scifact/queries.jsonl","r",encoding="utf-8") as f:
    for line in f:
        query.append(json.loads(line))

In [ ]:
print(query[4])

In [ ]:
query_message=query[0]["text"]

In [ ]:
response=retriever.invoke(query_message)
print(response)

In [ ]:
recieved_doc=response[0]
print(recieved_doc.metadata["corpus_id"])

In [ ]:
def is_present(corpus_id_list,response):
    found=0
    for id in corpus_id_list:
        for i in range(0,len(response)):
            recieved_doc_id=response[i].metadata["corpus_id"]
            if id==recieved_doc_id:
                found=found+1
                break
    return found/len(corpus_id_list)



In [ ]:
def Recall(K):
    retriever=Dense_vs.as_retriever(search_kwargs={"k": K})
    total_recall=0
    total=0
    for q in query:
        if q["_id"] not in qrels:
            continue
        corpus_Id_list=qrels[q["_id"]]
        response=retriever.invoke(q["text"])
        total=total+1
        total_recall=total_recall+is_present(corpus_Id_list,response)
    return total_recall/total

for k in [2,3,4,5,6]:
    recall=Recall(k)
    print("Recall@",k,": ",recall)


In [ ]:
print(len(response))

In [ ]:
print(response)

In [ ]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever1 = BM25Retriever.from_documents(docs, k=6)


In [ ]:
def tokenize(text):
    text=text.lower()
    for ch in ".,/;:\"|%-()[]'?":
        text=text.replace(ch," ")
    return text.split()

In [ ]:
bm25_retriever2 = BM25Retriever.from_documents(docs, preprocess_func=tokenize, k=6)


In [ ]:
bm25_retriever2.invoke(query[0]["text"])

In [ ]:
def Recall_on_BM25(retriever):
    total=0
    total_recall=0
    for q in query:
        if q["_id"] not in qrels:
            continue
        corpus_Id_list=qrels[q["_id"]]
        response=retriever.invoke(q["text"])
        total=total+1
        total_recall=total_recall+is_present(corpus_Id_list,response)

    return total_recall/total



In [ ]:
for kval in [2,3,4,5,6,7,8,9,10]:
    bm25_retriever=BM25Retriever.from_documents(docs,preprocess_func=tokenize,k=kval)
    recall=Recall_on_BM25(bm25_retriever)
    print("BM25-->Recall@",kval,": ",recall)

### Combining the dense and sparse vectors

In [ ]:
bm25_pool = BM25Retriever.from_documents(docs, preprocess_func=tokenize, k=30)
dense_pool = Dense_vs.as_retriever(search_kwargs={"k": 30})

hybrid = EnsembleRetriever(retrievers=[bm25_pool, dense_pool], weights=[0.5, 0.5])

response = hybrid.invoke(query[0]["text"])[:6]


In [ ]:
print(query[0])

In [ ]:
hybrid_response={}
for q in query:
    if q["_id"] not in qrels:
        continue
    hybrid_response[q["_id"]]=hybrid.invoke(q["text"])


In [ ]:
def Hybrid_Recall(kval):   
    total=0
    total_recall=0
    for q in hybrid_response:
        corpus_list=qrels[q]
        response=hybrid_response[q][:kval]
        total=total+1
        total_recall=total_recall+is_present(corpus_list,response)
    return total_recall/total


In [ ]:
for kval in [2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20]:
    recall=Hybrid_Recall(kval)
    print("Hybrid_recall@",kval,": ",recall)

### Using Reranker for better results


In [ ]:
reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [ ]:
qid=list(hybrid_response)[0]

In [ ]:
POOL=20

query_text={}
for q in query:
    query_text[q["_id"]]=q["text"]

def get_score(item):
    return item[0]
reranked_response={}

for qid in hybrid_response:
    candidates=hybrid_response[qid][:POOL]
    pairs=[]
    for doc in candidates:
        pairs.append((query_text[qid],doc.page_content))
    score=reranker.predict(pairs)
    scored=[]
    for i in range(len(candidates)):
        scored.append((score[i],candidates[i]))

    scored.sort(key=get_score,reverse=True)

    new_order=[]
    for s,doc in scored:
        new_order.append(doc)
    reranked_response[qid]=new_order

In [ ]:
reranked_response[qid]

In [ ]:

def reranker_Recall(kval):
    total=0
    total_recall=0
    for qid in reranked_response:
        corpus_list=qrels[qid]
        response=reranked_response[qid][:kval]
        total=total+1
        total_recall=total_recall + is_present(corpus_list,response)
    return total_recall/total

In [ ]:
for k in [2,3,4,5,6,7,8,9,10,11,12]:
    recall=reranker_Recall(k)
    print("Reranker_Recall@",k,": ",recall)

In [ ]:
recall=reranker_Recall(20)
print("Reranker_Recall@",20,": ",recall)

### Trying different Reranker

In [ ]:
reranker = CrossEncoder("BAAI/bge-reranker-base")


In [ ]:
POOL=10

query_text={}
for q in query:
    query_text[q["_id"]]=q["text"]

def get_score(item):
    return item[0]
reranked_response={}

for qid in hybrid_response:
    candidates=hybrid_response[qid][:POOL]
    pairs=[]
    for doc in candidates:
        pairs.append((query_text[qid],doc.page_content))
    score=reranker.predict(pairs)
    scored=[]
    for i in range(len(candidates)):
        scored.append((score[i],candidates[i]))

    scored.sort(key=get_score,reverse=True)

    new_order=[]
    for s,doc in scored:
        new_order.append(doc)
    reranked_response[qid]=new_order

In [ ]:

def reranker_Recall(kval):
    total=0
    total_recall=0
    for qid in reranked_response:
        corpus_list=qrels[qid]
        response=reranked_response[qid][:kval]
        total=total+1
        total_recall=total_recall + is_present(corpus_list,response)
    return total_recall/total

In [ ]:
for k in [2,3,4,5,6,7,8,9,10]:
    recall=reranker_Recall(k)
    print("Reranker_Recall@",k,": ",recall)

### Best response is seems to be Hybrid(RRF) + bge-reranker-base

In [ ]:
response = hybrid.invoke(query[0]["text"])[:10]

In [ ]:
print(response)

In [ ]:
qa_message=""" 
You are a scientific fact-checking assistant.
Using the below context, answer the user's claim is VALID/INVALID with the 
with few lines of information about it. Also give the citation ID.
For the claims outside of the given context, simply say NOT ENOUGH INFO PRESENT.
Do not use any knowledge outside the context.

context: {context}

"""

qa_prompt=ChatPromptTemplate.from_messages([
    ('system',qa_message),
    ('human', '{input}')
])

In [ ]:
llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
def get_score(item):
    return item[0]

POOL=10
topK=6

def retrieve_docs(x):
    question=x["input"]
    doc_base=hybrid.invoke(question)[:POOL]
    pairs=[]
    for doc in doc_base:
        pairs.append((question,doc.page_content))

    scores=reranker.predict(pairs)
    scored=[]
    for i in range(0,len(doc_base)):
        scored.append((scores[i],doc_base[i]))
    scored.sort(key=get_score,reverse=True)
    reranked_list=[]
    for s,doc in scored[:topK]:
        reranked_list.append(doc)
    return reranked_list

reranked_retriever=RunnableLambda(retrieve_docs)

In [ ]:
from langchain_core.prompts import PromptTemplate

doc_prompt =PromptTemplate.from_template("[{corpus_id}] {page_content}")
qa_chain=create_stuff_documents_chain(llm,qa_prompt,document_prompt=doc_prompt)
rag_chain=create_retrieval_chain(reranked_retriever,qa_chain)

In [ ]:
input="What is photosynthesis?"

In [ ]:
result=rag_chain.invoke({"input":"{input}"})
print(result["answer"])